In [ ]:
import random
import time
import gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from gym import Env
from torch.distributions import Categorical

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=

In [ ]:
def make_cartpole_env(render: bool = False) -> Env:
    env_id = "CartPole-v1"
    if not render:
        env = gym.make(env_id)
        return env
    try:
        env = gym.make(env_id, render_mode="human")
    except TypeError:
        env = gym.make(env_id)
    return env

In [ ]:
def reset_env(env: Env, seed: int | None = None):
    if seed is not None:
        try:
            state = env.reset(seed=seed)
        except TypeError:
            env.seed(seed)
            state = env.reset()
    else:
        state = env.reset()
    if isinstance(state, tuple):
        state, info = state
    return state

In [ ]:
def step_env(env: Env, action):
    out = env.step(action)
    if len(out) == 5:
        state, reward, terminated, truncated, info = out
        done = terminated or truncated
    else:
        state, reward, done, info = out
    return state, reward, done, info

In [ ]:
def get_action_probs(policy_net: PolicyNet, state_np: np.ndarray):
    state_tensor = torch.from_numpy(state_np)
    act_scores = policy_net(state_tensor)
    act_probs = F.softmax(act_scores, dim=-1)
    return act_probs

In [ ]:
def select_action(policy_net: PolicyNet, state_np: np.ndarray):
    probs = get_action_probs(policy_net, state_np)
    m = Categorical(probs)
    action = m.sample()
    log_prob = m.log_prob(action)
    return action.item(), log_prob

In [ ]:
def compute_returns(rewards, gamma):
    returns = []
    R = 0
    for r in rewards[::-1]:
        R = r + gamma * R
        returns.insert(0, R)
    return returns

In [ ]:
if __name__ == "__main__":
    trained_policy = train_cartpole()
    watch_agent_gui(trained_policy, num_episodes=3)

In [ ]:
class PolicyNet(nn.Module):
  def __init__(self, state_dim, action_dim, hidden_dim):
      super().__init__()
      self.lin1 = nn.Linear(self.state_dim, self.hidden_dim)
      self.relu1 = nn.ReLU()
      self.lin2 = nn.Linear(self.hidden_dim, self.action_dim)
  def forward(self, state:torch.Tensor) -> torch.Tensor:
      x = self.lin1(state)
      x = self.relu1(x)
      logits = self.lin2(x)
      return logits

In [ ]:
def train_cartpole(gamma=0.99, num_episodes=1000, hidden_dim=128, lr=1e-2, seed=0):
    set_seed(seed)
    env = make_cartpole_env(render=False)
    state_dim = env.observation_space.shape[0] # 4
    action_dim = env.action_space.n # 2
    policy_net = PolicyNet(state_dim, action_dim, hidden_dim)
    optimizer = optim.Adam(policy_net.parameters(), lr=lr)

    episode_rewards = [] # total reward per episode
    for episode in range(1, num_episodes + 1):
        state = reset_env(env, seed + episode)
        log_probs = []
        rewards = []
        done = False
        while not done:
            action, log_prob = select_action(policy_net, state)
            state, reward, done, info = step_env(env, action)
            log_probs.append(log_prob)
            rewards.append(reward)
returns = compute_returns(rewards, gamma)
returns = torch.tensor(returns)
returns = (returns - returns.mean()) / (returns.std() + 1e-8)